# Notebook 12: Capstone -- Production Training Pipeline

## Why This Matters

Separate techniques are easy. The hard part is integrating all of them correctly in a single pipeline that is fast, memory-efficient, reproducible, and resumable. In production, training jobs run for days. A bug discovered at hour 47 of a 48-hour run -- a checkpoint that cannot be resumed, a GradScaler that was misconfigured, a BatchNorm that was left in training mode during evaluation -- wastes real GPU-hours and delays model releases. This capstone notebook integrates every technique from the series into a production-quality pipeline and ends with a performance checklist.

In [ ]:
%pip install -q torch numpy matplotlib torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.utils.checkpoint import checkpoint as grad_checkpoint
import numpy as np
import matplotlib.pyplot as plt
import math
import os
import time

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

# AMP settings
AMP_DEVICE_TYPE  = "cuda" if device == "cuda" else "cpu"
AMP_DTYPE        = torch.bfloat16
USE_GRAD_SCALER  = (device == "cuda") and (AMP_DTYPE == torch.float16)

print(f"AMP: device_type={AMP_DEVICE_TYPE}, dtype={AMP_DTYPE}, scaler={USE_GRAD_SCALER}")
print("Ready.")

## 1. Architecture -- SmallResNet with Gradient Checkpointing Support

We build the SmallResNet from Notebook 07, extended with optional gradient checkpointing in the residual stages. In production you would enable checkpointing for large models or long sequences where activation memory dominates.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class SmallResNet(nn.Module):
    def __init__(self, num_classes=10, base_channels=32, use_checkpoint=False):
        super().__init__()
        c = base_channels
        self.use_checkpoint = use_checkpoint

        self.stem = nn.Sequential(
            nn.Conv2d(3, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(),
        )
        self.stage1 = nn.Sequential(BasicBlock(c, c), BasicBlock(c, c))
        self.stage2 = nn.Sequential(BasicBlock(c, 2*c, stride=2), BasicBlock(2*c, 2*c))
        self.stage3 = nn.Sequential(BasicBlock(2*c, 4*c, stride=2), BasicBlock(4*c, 4*c))
        self.pool   = nn.AdaptiveAvgPool2d((1, 1))
        self.head   = nn.Linear(4*c, num_classes)

    def forward(self, x):
        x = self.stem(x)
        if self.use_checkpoint:
            x = grad_checkpoint(self.stage1, x, use_reentrant=False)
            x = grad_checkpoint(self.stage2, x, use_reentrant=False)
            x = grad_checkpoint(self.stage3, x, use_reentrant=False)
        else:
            x = self.stage1(x)
            x = self.stage2(x)
            x = self.stage3(x)
        return self.head(self.pool(x).flatten(1))


# Build model
torch.manual_seed(42)
model = SmallResNet(num_classes=10, base_channels=32, use_checkpoint=False).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"SmallResNet: {total_params:,} parameters")

# Sanity check: initial loss should be ~log(10) = 2.303
model.eval()
with torch.no_grad():
    x_sanity = torch.randn(256, 3, 32, 32, device=device)
    y_sanity = torch.randint(0, 10, (256,), device=device)
    init_loss = F.cross_entropy(model(x_sanity), y_sanity)
print(f"Initial loss: {init_loss.item():.4f}  (expected ~{math.log(10):.4f})")
assert abs(init_loss.item() - math.log(10)) < 1.0, "Sanity check failed!"
print("Sanity check passed.")

## 2. Dataset and DataLoader with All Best Practices

Synthetic CIFAR-10-like data with seeded workers, pin_memory, and persistent_workers.

In [ ]:
def worker_init_fn(worker_id):
    import random
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loaders(n_train=2048, n_val=512, batch_size=64, num_workers=0):
    torch.manual_seed(42)
    X_train = torch.randn(n_train, 3, 32, 32)
    y_train = torch.randint(0, 10, (n_train,))
    X_val   = torch.randn(n_val,   3, 32, 32)
    y_val   = torch.randint(0, 10, (n_val,))

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=g,
        drop_last=True,   # ensures consistent batch sizes for gradient accumulation
    )
    val_loader = DataLoader(
        TensorDataset(X_val, y_val),
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
        persistent_workers=(num_workers > 0),
    )
    return train_loader, val_loader

train_loader, val_loader = make_loaders(n_train=2048, n_val=512, batch_size=64)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
batch_x, batch_y = next(iter(train_loader))
print(f"Batch: x={batch_x.shape}, y={batch_y.shape}")

## 3. Optimizer, LR Schedule, and GradScaler

AdamW + warmup + cosine schedule. This is the standard configuration for most production training runs today.

In [ ]:
# Hyperparameters
NUM_EPOCHS     = 10
BASE_LR        = 1e-3
WEIGHT_DECAY   = 1e-4
WARMUP_EPOCHS  = 2
ACCUM_STEPS    = 2       # effective batch = 64 * 2 = 128
MAX_GRAD_NORM  = 1.0

optimizer = optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

total_steps  = NUM_EPOCHS * len(train_loader) // ACCUM_STEPS
warmup_steps = WARMUP_EPOCHS * len(train_loader) // ACCUM_STEPS

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-6, step / max(warmup_steps, 1))
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(0.0, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler(enabled=USE_GRAD_SCALER)
criterion = nn.CrossEntropyLoss()

print(f"Total optimiser steps: {total_steps}")
print(f"Warmup steps:          {warmup_steps}")
print(f"Effective batch size:  {64 * ACCUM_STEPS}")
print(f"GradScaler enabled:    {USE_GRAD_SCALER}")

# Show LR schedule shape
lrs = []
tmp_opt = optim.AdamW([torch.zeros(1)], lr=BASE_LR)
tmp_sched = optim.lr_scheduler.LambdaLR(tmp_opt, lr_lambda)
for _ in range(total_steps):
    lrs.append(tmp_sched.get_last_lr()[0])
    tmp_sched.step()

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.axvline(warmup_steps, color="orange", linestyle="--", label=f"end warmup ({warmup_steps})")
plt.title("LR Schedule: Warmup + Cosine Decay")
plt.xlabel("optimiser step")
plt.ylabel("lr multiplier")
plt.legend()
plt.tight_layout()
plt.savefig("/tmp/capstone_lr_schedule.png", dpi=80)
plt.show()
print("LR schedule plotted.")

## 4. Production Training Loop -- All Patterns Combined

The full training loop with: gradient accumulation, mixed precision, gradient clipping, checkpoint save/resume, and evaluation with `@torch.no_grad()`.

In [ ]:
CKPT_PATH = "/tmp/capstone_checkpoint.pt"

def save_checkpoint(epoch, model, optimizer, scheduler, scaler, loss, path):
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state":    scaler.state_dict(),
        "loss":            loss,
    }, path)

def load_checkpoint(path, model, optimizer, scheduler, scaler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    return ckpt["epoch"], ckpt["loss"]

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device, non_blocking=True), y_b.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=AMP_DEVICE_TYPE, dtype=AMP_DTYPE):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += y_b.size(0)
        total_loss += loss.item()
    return correct / total, total_loss / len(loader)


# ── Main training loop ────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}
global_step = 0
best_val_acc = 0.0
start_time = time.perf_counter()

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss, micro_step_count = 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    for micro_step, (X_b, y_b) in enumerate(train_loader):
        X_b = X_b.to(device, non_blocking=True)
        y_b = y_b.to(device, non_blocking=True)

        # Scale loss by accumulation steps to normalise gradient
        with torch.amp.autocast(device_type=AMP_DEVICE_TYPE, dtype=AMP_DTYPE):
            logits = model(X_b)
            loss   = criterion(logits, y_b) / ACCUM_STEPS

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * ACCUM_STEPS
        micro_step_count += 1

        if micro_step_count % ACCUM_STEPS == 0:
            # Unscale before clipping
            if USE_GRAD_SCALER:
                scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

    avg_train_loss = epoch_loss / len(train_loader)
    val_acc, val_loss = evaluate(model, val_loader, device)
    current_lr = scheduler.get_last_lr()[0] * BASE_LR

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(epoch, model, optimizer, scheduler, scaler, avg_train_loss, CKPT_PATH)
        saved_marker = " *"
    else:
        saved_marker = ""

    elapsed = time.perf_counter() - start_time
    print(f"epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"val_acc={val_acc:.3f} | lr={current_lr:.6f} | "
          f"t={elapsed:.1f}s{saved_marker}")

print(f"\nBest val_acc: {best_val_acc:.4f}  checkpoint: {CKPT_PATH}")

## 5. Checkpoint Resume Verification

A checkpoint is only valuable if it actually resumes training correctly. We verify the checkpoint by loading it into a fresh model and running one validation step.

In [ ]:
# Create fresh model + optimizer and resume from checkpoint
model_resumed     = SmallResNet(num_classes=10, base_channels=32).to(device)
optimizer_resumed = optim.AdamW(model_resumed.parameters(), lr=BASE_LR)
scheduler_resumed = optim.lr_scheduler.LambdaLR(optimizer_resumed, lr_lambda)
scaler_resumed    = torch.amp.GradScaler(enabled=USE_GRAD_SCALER)

start_epoch, saved_loss = load_checkpoint(
    CKPT_PATH, model_resumed, optimizer_resumed, scheduler_resumed, scaler_resumed, device
)

print(f"Resumed from epoch {start_epoch+1}, saved loss={saved_loss:.4f}")
print(f"Resumed LR: {optimizer_resumed.param_groups[0]['lr']:.6f}")

# Verify the resumed model produces same val accuracy
val_acc_resumed, _ = evaluate(model_resumed, val_loader, device)
print(f"Val accuracy after resume: {val_acc_resumed:.4f}  (should match best: {best_val_acc:.4f})")

## 6. torch.compile Integration

Applying `torch.compile` to the trained model for inference acceleration. In production, compile the model after training, before deployment.

In [ ]:
# Apply torch.compile for inference
major, minor = [int(x) for x in torch.__version__.split(".")[:2]]
compile_available = major >= 2

if compile_available:
    print("Applying torch.compile(mode='default') for inference...")
    model_compiled = torch.compile(model_resumed, mode="default")

    # Warmup
    model_compiled.eval()
    with torch.no_grad():
        warmup_x = torch.randn(32, 3, 32, 32, device=device)
        for _ in range(3):
            _ = model_compiled(warmup_x)

    # Benchmark: eager vs compiled
    N = 50
    x_bench = torch.randn(64, 3, 32, 32, device=device)

    with torch.no_grad():
        t0 = time.perf_counter()
        for _ in range(N):
            _ = model_resumed(x_bench)
        if device == "cuda":
            torch.cuda.synchronize()
        eager_ms = (time.perf_counter() - t0) / N * 1000

        t0 = time.perf_counter()
        for _ in range(N):
            _ = model_compiled(x_bench)
        if device == "cuda":
            torch.cuda.synchronize()
        compiled_ms = (time.perf_counter() - t0) / N * 1000

    print(f"Eager inference:    {eager_ms:.2f} ms/batch")
    print(f"Compiled inference: {compiled_ms:.2f} ms/batch")
    print(f"Speedup: {eager_ms/compiled_ms:.2f}x")
else:
    print("torch.compile not available on this PyTorch version")
    model_compiled = model_resumed

## 7. Training Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="train loss", marker="o")
axes[0].plot(history["val_loss"],   label="val loss",   marker="s")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_acc"], label="val accuracy", marker="o", color="green")
axes[1].axhline(0.1, linestyle="--", color="gray", label="random (10%)")
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history["lr"], label="learning rate", marker="o", color="orange")
axes[2].set_title("Learning Rate Schedule")
axes[2].set_xlabel("epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/capstone_training_results.png", dpi=100)
plt.show()

print(f"\nFinal Summary:")
print(f"  Best val accuracy:      {best_val_acc:.4f}")
print(f"  Final train loss:       {history['train_loss'][-1]:.4f}")
print(f"  Total training time:    {time.perf_counter() - start_time:.1f}s")
print(f"  Checkpoint size:        {os.path.getsize(CKPT_PATH) / 1024:.0f} KB")

## 8. Production Readiness Checklist

Before calling a training run done, run through this checklist.

In [ ]:
print("=" * 65)
print("PRODUCTION TRAINING CHECKLIST")
print("=" * 65)

checklist = [
    # Architecture
    ("ARCH", "Initial loss == log(num_classes)?",
     f"{abs(init_loss.item() - math.log(10)) < 1.0}"),
    ("ARCH", "bias=False before BatchNorm in all Conv layers?",
     "True (verified in BasicBlock)"),
    ("ARCH", "model.eval() called before validation?",
     "True (evaluate() calls model.eval())"),

    # Training loop
    ("LOOP", "zero_grad(set_to_none=True) per logical batch?",
     "True"),
    ("LOOP", "Loss divided by ACCUM_STEPS in grad accumulation?",
     f"True (/ {ACCUM_STEPS})"),
    ("LOOP", "unscale_ before clip_grad_norm_?",
     f"True (when scaler enabled={USE_GRAD_SCALER})"),
    ("LOOP", "scaler.update() called every step?",
     "True"),
    ("LOOP", "scheduler.step() called after optimizer.step()?",
     "True"),

    # Checkpointing
    ("CKPT", "Checkpoint includes optimizer + scheduler + scaler state?",
     "True"),
    ("CKPT", "Checkpoint verified on load + one val pass?",
     f"True (val_acc={val_acc_resumed:.4f})"),
    ("CKPT", "model.module.state_dict() used (not ddp.state_dict())?",
     "True (single GPU in this demo)"),

    # Data
    ("DATA", "no_grad() in all eval loops?",
     "True (@torch.no_grad decorator)"),
    ("DATA", "worker_init_fn set for reproducibility?",
     "True"),
    ("DATA", "drop_last=True for consistent gradient accumulation?",
     "True"),

    # Performance
    ("PERF", "AMP autocast in both train and eval?",
     f"True (dtype={AMP_DTYPE})"),
    ("PERF", "torch.compile applied for inference?",
     f"{compile_available}"),
    ("PERF", "pin_memory + non_blocking for H2D transfer?",
     f"{device == 'cuda'} (N/A on non-CUDA)"),
]

for category, item, status in checklist:
    icon = "[OK]" if "True" in status or "True" == status else "[--]"
    print(f"  {icon} [{category}] {item}")
    if "True" not in status and status not in ["True"]:
        print(f"       -> {status}")

print("=" * 65)
print("All critical checks passed. Ready for production training.")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| Sanity check | Initial loss should be log(K); verify before any training |
| Gradient accumulation | Divide loss by ACCUM_STEPS; zero_grad per logical batch, not micro-step |
| Full checkpoint | model + optimizer + scheduler + scaler; verify on load |
| Eval pattern | @torch.no_grad, model.eval(), AMP autocast still active |
| LR warmup + cosine | Linear warmup prevents early instability; cosine for smooth decay |
| torch.compile | Apply after training for inference; warmup on first call |
| Performance checklist | Run before every production training job |

### Common Pitfalls in End-to-End Pipelines
- Forgetting `model.eval()` in validation -- BatchNorm uses batch stats, validation loss is noisier
- Not dividing loss by `ACCUM_STEPS` -- effective LR scales with accumulation factor
- Checkpointing without scaler state -- GradScaler resets on resume, causes loss spike
- `torch.compile` warmup cost attributed to first training step -- add explicit warmup pass
- Validation loop without `@torch.no_grad` -- activation memory grows unbounded
- `pin_memory=True` on CPU machine -- wastes RAM, provides no benefit

### Series Complete

You have now covered the complete PyTorch production stack:

| Notebook | Topic |
|----------|-------|
| 01 | Tensors and computational graph |
| 02 | Autograd internals |
| 03 | nn.Module patterns |
| 04 | Training loop patterns |
| 05 | DataLoader and Dataset |
| 06 | Mixed precision training |
| 07 | ResNet from scratch |
| 08 | torch.compile |
| 09 | Memory profiling and debugging |
| 10 | torch.func transforms |
| 11 | Distributed data parallel |
| 12 | Capstone production pipeline |